# MovieLens: Spark + Polaris + SeaweedFS

Read existing bronze CSV, write Iceberg silver, and create a gold movie report. Spark executes SQL, Polaris catalogs Iceberg tables, and SeaweedFS stores the files.

```text
Spark ── REST catalog ──> Polaris: test_catalog
  ├── S3A reads ────────> SeaweedFS: datalake/movielens/bronze/
  └── Iceberg S3FileIO ─> SeaweedFS: catalog-configured silver/gold storage
```

Use a fresh kernel, Spark 3.5.x / Scala 2.12 and Java 17. This example fetches Iceberg 1.9.2 and Hadoop AWS 3.3.4 from Maven; match the Hadoop connector to your Spark distribution.

use Polaris `8181` with `root:s3cr3t`, existing catalog `test_catalog`, and SeaweedFS `9001` with `team:team1234`. Edit the settings if you use Polaris `8880` or a different principal. The catalog must already exist, with namespace/table read/write permissions. If the notes' cleanup deleted it, recreate it through the Polaris Management API first.

With multiple Spark hosts or containers, replace localhost with an endpoint reachable by the driver and all workers. Polaris inside Docker uses its separately configured internal endpoint, such as `http://seaweedfs:8333`.


## 1. Spark configuration

S3A reads bronze CSV; Iceberg S3FileIO reads/writes table files. The Polaris `warehouse` setting is the existing server catalog name, not an S3 path.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
polaris_url = os.environ.get("POLARIS_BASE_URL", "http://localhost:8181").rstrip("/")
catalog_name = os.environ.get("POLARIS_CATALOG", "test_catalog")
polaris_credential = os.environ.get("POLARIS_CREDENTIAL", "root:s3cr3t")
s3_endpoint = os.environ.get("S3_ENDPOINT", "http://localhost:9001")

spark = (
    SparkSession.builder
    .appName("MovieLens-Polaris-SeaweedFS")
    .master(master_url)
    .config("spark.jars.packages", ",".join([
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2",
        "org.apache.iceberg:iceberg-aws-bundle:1.9.2",
        "org.apache.hadoop:hadoop-aws:3.3.4",
    ]))
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    # SeaweedFS S3A: read existing bronze files
    .config("spark.hadoop.fs.s3a.endpoint", s3_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", "team")
    .config("spark.hadoop.fs.s3a.secret.key", "team1234")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

    # Polaris Iceberg REST catalog
    .config("spark.sql.catalog.polaris", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.type", "rest")
    .config("spark.sql.catalog.polaris.uri", f"{polaris_url}/api/catalog")
    .config("spark.sql.catalog.polaris.warehouse", catalog_name)
    .config("spark.sql.catalog.polaris.oauth2-server-uri", f"{polaris_url}/api/catalog/v1/oauth/tokens")
    .config("spark.sql.catalog.polaris.credential", polaris_credential)
    .config("spark.sql.catalog.polaris.scope", "PRINCIPAL_ROLE:ALL")

    # Iceberg storage: direct credentials, no STS credential vending
    .config("spark.sql.catalog.polaris.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.polaris.s3.endpoint", s3_endpoint)
    .config("spark.sql.catalog.polaris.s3.path-style-access", "true")
    .config("spark.sql.catalog.polaris.s3.access-key-id", "team")
    .config("spark.sql.catalog.polaris.s3.secret-access-key", "team1234")
    .config("spark.sql.catalog.polaris.client.region", "us-east-1")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Read bronze data

Raw CSV files use `s3a://`. These are not Iceberg tables; bronze files are retained unchanged.

In [ ]:
movies_path = "s3a://datalake/movielens/bronze/movies/movies.csv"
ratings_path = "s3a://datalake/movielens/bronze/ratings/ratings.csv"

movies_df = spark.read.option("header", "true").schema(
    "movieId INT, title STRING, genres STRING"
).csv(movies_path)
ratings_df = spark.read.option("header", "true").schema(
    "userId INT, movieId INT, rating DOUBLE, timestamp LONG"
).csv(ratings_path)

movies_df.printSchema()
movies_df.show(10, truncate=False)
ratings_df.show(10, truncate=False)
print("Number of movies:", movies_df.count())
print("Number of ratings:", ratings_df.count())

## 3. List and create namespaces

`polaris` is the Spark alias for `test_catalog`. Namespace creation happens inside this existing catalog.

In [ ]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW NAMESPACES IN polaris").show()
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.gold")

## 4. Prepare silver DataFrames

Keep valid IDs, deduplicate movies by ID and rating events by user/movie/timestamp, and keep ratings in the MovieLens 0.5–5.0 range.

In [ ]:
silver_movies_df = movies_df.filter(
    "movieId IS NOT NULL AND title IS NOT NULL"
).dropDuplicates(["movieId"])

silver_ratings_df = (
    ratings_df.filter("userId IS NOT NULL AND movieId IS NOT NULL AND timestamp IS NOT NULL")
    .filter(F.col("rating").between(0.5, 5.0))
    .dropDuplicates(["userId", "movieId", "timestamp"])
)

## 5. Write silver Iceberg tables

`createOrReplace()` creates the demo table or replaces its current contents on rerun. Bronze is untouched. Iceberg locations come from the catalog's allowed base location:  configured to `s3://polaris-test/`. To put silver/gold in `datalake` instead, first provision a catalog allowing that bucket; changing the bronze read path does not change the catalog's storage location.

In [ ]:
silver_movies_df.writeTo("polaris.silver.movielens_movies").using("iceberg").createOrReplace()
silver_ratings_df.writeTo("polaris.silver.movielens_ratings").using("iceberg").createOrReplace()

spark.sql("SHOW TABLES IN polaris.silver").show()
spark.sql("DESCRIBE TABLE polaris.silver.movielens_movies").show(truncate=False)
spark.sql("SELECT * FROM polaris.silver.movielens_movies LIMIT 10").show(truncate=False)

## 6. Gold movie analytics

Keep movies with **at least 100 ratings** and an **average rating of at least 4.0**, then join movie titles and genres. Filter on the unrounded average.

In [ ]:
gold_df = spark.sql("""
    SELECT m.movieId, m.title, m.genres, r.rating_count, r.avg_rating
    FROM polaris.silver.movielens_movies m
    JOIN (
        SELECT movieId, COUNT(*) AS rating_count, AVG(rating) AS avg_rating
        FROM polaris.silver.movielens_ratings
        GROUP BY movieId
        HAVING COUNT(*) >= 100 AND AVG(rating) >= 4.0
    ) r ON m.movieId = r.movieId
""")
gold_df.writeTo("polaris.gold.movielens_top_movies").using("iceberg").createOrReplace()

## 7. List, describe and query gold

In [ ]:
spark.sql("SHOW TABLES IN polaris.gold").show()
spark.sql("DESCRIBE EXTENDED polaris.gold.movielens_top_movies").show(truncate=False)
spark.sql("""
    SELECT title, genres, rating_count, ROUND(avg_rating, 2) AS avg_rating
    FROM polaris.gold.movielens_top_movies
    ORDER BY avg_rating DESC, rating_count DESC, title
    LIMIT 20
""").show(truncate=False)

## 8. Inspect Iceberg snapshots

Snapshots record committed table versions; Polaris tracks catalog metadata while SeaweedFS stores the table files.

In [ ]:
spark.sql("""
    SELECT committed_at, snapshot_id, operation
    FROM polaris.gold.movielens_top_movies.snapshots
    ORDER BY committed_at DESC
""").show(truncate=False)

## Finish

Stop Spark when done; tables remain available. If startup fails, restart the kernel after changing JARs/settings, check the catalog and grants, match the Hadoop connector version, and check endpoint reachability from workers.

References: [Iceberg Spark configuration](https://iceberg.apache.org/docs/1.9.2/spark-configuration/), [Iceberg Spark writes](https://iceberg.apache.org/docs/1.9.2/spark-writes/), [Apache Polaris](https://polaris.apache.org/).

Notebook structure and Python syntax checked offline. Live Spark/Polaris/SeaweedFS execution has not been verified here.

In [ ]:
spark.stop()